# Notebook 1: Data Loading and Memory Management

This notebook covers **Section 1** of the assignment:
- Demonstrate the memory-optimised loading strategy
- Produce before/after memory evidence
- Load all daily files and build the processed traffic matrix
- Save the result as a compressed Parquet file for reuse

In [1]:
import sys
sys.path.insert(0, '../src')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_loader import (
    memory_optimisation_demo,
    load_all_files,
    build_traffic_matrix,
    save_processed,
    load_processed,
    _mem_mb,
)

DATA_DIR = r'C:\Users\LENOVO\milan_data'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1.1 Memory Optimisation Demo

We compare naive loading (all 8 columns, default dtypes) vs. our optimised approach
(3 columns, downcast dtypes, immediate aggregation per file).

In [2]:
# Pick one sample file for the demo
import glob
sample_file = sorted(glob.glob(os.path.join(DATA_DIR, '*.txt')))[0]
print(f'Demo file: {sample_file}')

stats = memory_optimisation_demo(sample_file)

Demo file: C:\Users\LENOVO\milan_data\sms-call-internet-mi-2013-11-01.txt
Memory Optimisation Demo
  Naive   peak memory : 332.7 MB  (shape (4842625, 8))
  Optimised peak memory: 276.8 MB  (shape (1439982, 3))
  Reduction factor    : 1.2×


## 1.2 Load All Files

In [3]:
print(f'Memory before loading all files: {_mem_mb():.1f} MB')
df_long = load_all_files(DATA_DIR, verbose=True)
print(f'\nMemory after loading all files: {_mem_mb():.1f} MB')

Memory before loading all files: 151.2 MB
Found 62 files.
Memory before loading: 151.2 MB
  [01/62] sms-call-internet-mi-2013-11-01.txt  rows=1,439,982  mem=22.0MB  time=3.7s
  [02/62] sms-call-internet-mi-2013-11-02.txt  rows=1,439,986  mem=22.0MB  time=3.3s
  [03/62] sms-call-internet-mi-2013-11-03.txt  rows=1,439,963  mem=22.0MB  time=3.4s
  [04/62] sms-call-internet-mi-2013-11-04.txt  rows=1,439,976  mem=22.0MB  time=4.1s
  [05/62] sms-call-internet-mi-2013-11-05.txt  rows=1,439,977  mem=22.0MB  time=3.9s
  [06/62] sms-call-internet-mi-2013-11-06.txt  rows=1,439,980  mem=22.0MB  time=3.9s
  [07/62] sms-call-internet-mi-2013-11-07.txt  rows=1,439,981  mem=22.0MB  time=4.1s
  [08/62] sms-call-internet-mi-2013-11-08.txt  rows=1,439,976  mem=22.0MB  time=4.0s
  [09/62] sms-call-internet-mi-2013-11-09.txt  rows=1,439,978  mem=22.0MB  time=3.5s
  [10/62] sms-call-internet-mi-2013-11-10.txt  rows=1,439,983  mem=22.0MB  time=3.3s
  [11/62] sms-call-internet-mi-2013-11-11.txt  rows=1,439,98

## 1.3 Pivot to Traffic Matrix

In [4]:
matrix = build_traffic_matrix(df_long, verbose=True)
print('\nMatrix preview:')
matrix.iloc[:3, :5]


Pivoting to (time × square) matrix …
  Matrix shape : (8928, 10000)
  Date range   : 2013-10-31 23:00:00+00:00 → 2014-01-01 22:50:00+00:00
  Memory before pivot: 1513.8 MB
  Memory after pivot : 717.5 MB
  Remaining NaN: 0

Matrix preview:


square_id,1,2,3,4,5
datetime,,,,,
2013-10-31 23:00:00+00:00,11.028366,11.058225,11.090008,10.941881,9.916549
2013-10-31 23:10:00+00:00,11.127101,11.167927,11.211384,11.008849,9.987806
2013-10-31 23:20:00+00:00,10.892771,10.915638,10.939980,10.826535,9.772990


In [9]:
matrix.columns = matrix.columns.astype(str)
matrix.to_parquet(out_path, compression=None, engine="fastparquet")
size_mb = os.path.getsize(out_path) / 1024 ** 2
print(f"Saved → {out_path}  ({size_mb:.1f} MB on disk)")

Saved → ../data/processed/traffic_matrix.parquet  (342.8 MB on disk)


In [10]:
# Verify it loads correctly
matrix_loaded = load_processed(out_path)
matrix_loaded.columns = matrix_loaded.columns.astype(int)
print('Date range:', matrix_loaded.index[0], '->', matrix_loaded.index[-1])
print('Shape:', matrix_loaded.shape)
print('Column dtype:', matrix_loaded.columns.dtype)

Loaded processed matrix: (8928, 10000)  from ../data/processed/traffic_matrix.parquet
Date range: 2013-10-31 23:00:00+00:00 -> 2014-01-01 22:50:00+00:00
Shape: (8928, 10000)
Column dtype: int64


## 1.4 Save Processed Matrix

In [11]:
out_path = os.path.join(PROCESSED_DIR, 'traffic_matrix.parquet')
save_processed(matrix, out_path)

Saved processed matrix → ../data/processed/traffic_matrix.parquet  (342.8 MB on disk)


## 1.5 Verify Reload

In [12]:
matrix_loaded = load_processed(out_path)
print('Date range:', matrix_loaded.index[0], '->', matrix_loaded.index[-1])
print('Shape:', matrix_loaded.shape)

Loaded processed matrix: (8928, 10000)  from ../data/processed/traffic_matrix.parquet
Date range: 2013-10-31 23:00:00+00:00 -> 2014-01-01 22:50:00+00:00
Shape: (8928, 10000)
